# Germany BAFA — Phase 0 exploration

Probe [Amtliche Mineralöldaten](https://www.bafa.de/DE/Energie/Rohstoffe/Mineraloelstatistik/mineraloel_node.html) before building a production scraper.

**Goals**

1. Understand **format eras** (XLSX vs PDF, CID-encoded PDFs).
2. Parse **demand** (Inlandsablieferungen), **stocks** (Eigentumsendbestand), **bio blends** (Beimischung).
3. See whether end-2019 → present is scrapeable enough for seasonality + diesel↔biodiesel substitution.

**Draft helpers:** [`reference/_germany_bafa_probe.py`](../reference/_germany_bafa_probe.py) (not production-wired yet).

**Decisions locked from planning**

- Scope: demand + bio + stocks
- History: from end-2019
- Substitution: fossil diesel vs FAME+HVO; gasoline vs ethanol
- §6 product totals are blended; §9 is blend *components* (do not double-count)
- DuckDB warehouse only after v1 parsers look solid

## 1. Setup

In [14]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px


def _resolve_project_root() -> Path:
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "reference" / "_germany_bafa_probe.py").exists():
            return candidate
        nested = candidate / "country_oil_scraper" / "reference" / "_germany_bafa_probe.py"
        if nested.exists():
            return nested.parent.parent
    raise FileNotFoundError("Could not find country_oil_scraper root")


PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analytics import seasonality_by_year_chart
from reference._germany_bafa_probe import (
    DISPLAY_LABELS,
    SEASONALITY_BIO_PRODUCTS,
    SEASONALITY_DEMAND_PRODUCTS,
    describe_file,
    download_many,
    download_month,
    month_grid,
    parse_probe_file,
    seasonality_chart_inputs,
)

PROBE_DIR = PROJECT_ROOT / "data" / "raw" / "germany" / "probe"
PROBE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_colwidth", 60)
print("Project root:", PROJECT_ROOT)
print("Probe dir   :", PROBE_DIR)
print("Demand seasonality panels:", len(SEASONALITY_DEMAND_PRODUCTS))
print("Bio seasonality panels   :", len(SEASONALITY_BIO_PRODUCTS))

Project root: c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper
Probe dir   : c:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\country_oil_scraper\data\raw\germany\probe
Demand seasonality panels: 14
Bio seasonality panels   : 6


## 2. What BAFA actually ships (format eras)

Landing pages in Infothek always look like PDFs, but the binary payload changes:

| Era (approx.) | Real format | Notes |
|---|---|---|
| ≤ early 2024 (many months) | **XLSX** | URL may still say `.pdf`. Sheets `Tab 6c` / `Tab 8` / `Tab 9` |
| Mid-2024 (e.g. 2024-06) | PDF, **CID fonts** | Text looks like `(cid:36)…`; decode with +29 offset |
| 2025+ | PDF, clean text | 11 pages; §6 demand, §8 stocks, §9 bio; HVO/FAME split |

Older Infothek slugs use German month names (`2019_dezember`); newer use `YYYY_MM`.

In [15]:
# Representative months across eras (end-2019 → latest)
ERA_MONTHS = [
    (2019, 12),
    (2020, 6),
    (2022, 1),
    (2023, 6),
    (2024, 1),
    (2024, 6),
    (2024, 12),
    (2025, 1),
    (2025, 8),
    (2025, 12),
]

era_files = download_many(ERA_MONTHS, PROBE_DIR)
era_info = pd.DataFrame([describe_file(f.path) | {"year": f.year, "month": f.month, "kind": f.kind} for f in era_files])
era_info[["year", "month", "kind", "bytes", "pages", "cid_encoded", "has_tab_6c", "has_tab_9"]].sort_values(["year", "month"])

,year,month,kind,bytes,pages,cid_encoded,has_tab_6c,has_tab_9
0,2019,12,xlsx,152937,NaN,NaN,True,True
1,2020,6,xlsx,153102,NaN,NaN,True,True
2,2022,1,xlsx,120775,NaN,NaN,True,True
3,2023,6,pdf,157508,24.0,False,NaN,NaN
4,2024,1,xlsx,119603,NaN,NaN,True,True
5,2024,6,pdf,196805,21.0,True,NaN,NaN
6,2024,12,pdf,156251,23.0,False,NaN,NaN
7,2025,1,pdf,104210,11.0,False,NaN,NaN
8,2025,8,pdf,105937,11.0,False,NaN,NaN
9,2025,12,pdf,106068,11.0,False,NaN,NaN


## 3. Parse one file per era

Draft parsers keep **source-native** German labels and tag:

- `TOTDEMO` — domestic deliveries (§6 / Tab 6c)
- `CLOSTLV` — closing stocks (§8 / Tab 8)
- `BIOBLEND` — bio blending components (§9 / Tab 9)

Units are **tonnes**. Only the monthly `Berichtsmonat` column is kept (YTD ignored).

In [16]:
parsed_parts = []
for probe in era_files:
    try:
        df = parse_probe_file(probe)
        parsed_parts.append(df)
        counts = df.groupby("metric_type").size().to_dict() if not df.empty else {}
        print(f"{probe.year}-{probe.month:02d} {probe.kind:4} rows={len(df):3} {counts}")
    except Exception as exc:
        print(f"{probe.year}-{probe.month:02d} FAILED: {exc}")

era_long = pd.concat(parsed_parts, ignore_index=True) if parsed_parts else pd.DataFrame()
print("\nTotal probe rows:", len(era_long))
era_long.groupby(["date", "metric_type"]).size().unstack(fill_value=0).sort_index()

2019-12 xlsx rows= 49 {'BIOBLEND': 3, 'CLOSTLV': 23, 'TOTDEMO': 23}
2020-06 xlsx rows= 49 {'BIOBLEND': 3, 'CLOSTLV': 23, 'TOTDEMO': 23}
2022-01 xlsx rows= 49 {'BIOBLEND': 3, 'CLOSTLV': 23, 'TOTDEMO': 23}
2023-06 pdf  rows= 51 {'BIOBLEND': 3, 'CLOSTLV': 25, 'TOTDEMO': 23}
2024-01 xlsx rows= 49 {'BIOBLEND': 3, 'CLOSTLV': 23, 'TOTDEMO': 23}
2024-06 pdf  rows= 52 {'BIOBLEND': 4, 'CLOSTLV': 25, 'TOTDEMO': 23}
2024-12 pdf  rows= 51 {'BIOBLEND': 3, 'CLOSTLV': 25, 'TOTDEMO': 23}
2025-01 pdf  rows= 36 {'BIOBLEND': 5, 'CLOSTLV': 13, 'TOTDEMO': 18}
2025-08 pdf  rows= 36 {'BIOBLEND': 5, 'CLOSTLV': 13, 'TOTDEMO': 18}
2025-12 pdf  rows= 33 {'BIOBLEND': 5, 'CLOSTLV': 11, 'TOTDEMO': 17}

Total probe rows: 455


metric_type,BIOBLEND,CLOSTLV,TOTDEMO
date,,,
2019-12-01,3,23,23
2020-06-01,3,23,23
2022-01-01,3,23,23
2023-06-01,3,25,23
2024-01-01,3,23,23
2024-06-01,4,25,23
2024-12-01,3,25,23
2025-01-01,5,13,18
2025-08-01,5,13,18


In [17]:
# Spot-check latest modern PDF vs an XLSX month
for label, key in [("xlsx 2020-06", (2020, 6)), ("pdf 2025-08", (2025, 8))]:
    sub = era_long[(era_long["date"].dt.year == key[0]) & (era_long["date"].dt.month == key[1])]
    print(f"\n=== {label} ===")
    print("Demand:")
    print(
        sub[sub["metric_type"] == "TOTDEMO"][["product_native", "value"]]
        .head(12)
        .to_string(index=False)
    )
    print("Bio:")
    print(
        sub[sub["metric_type"] == "BIOBLEND"][["product_native", "value"]].to_string(index=False)
    )


=== xlsx 2020-06 ===
Demand:
            product_native     value
                 Rohbenzin  886507.0
            Ottokraftstoff 1374656.0
         Benzinkomponenten  280964.0
          Dieselkraftstoff 2812365.0
            Heizöl, leicht 1266213.0
Mitteldestillatkomponenten   86529.0
            Heizöl, schwer   64764.0
            HS-Komponenten  124159.0
                Flüssiggas  249511.0
             Raffineriegas   33565.0
             Spezialbenzin   11796.0
                Testbenzin   10296.0
Bio:
            product_native     value
     Bioethanol an ETBE a)  12369.93
                Bioethanol  81232.00
Biodiesel (FAME), HVO, BTL 250600.00

=== pdf 2025-08 ===
Demand:
                     product_native     value
                          Rohbenzin 1105588.0
          Raffinerieeinsatzmaterial   77075.0
                   Dieselkraftstoff 2814181.0
                     Heizöl, leicht  681965.0
                     Heizöl, schwer   90306.0
                         Flüssi

### Known quirks seen in the probe

- **Confidentiality blanks:** e.g. Aug 2025 `Ottokraftstoff` missing in §6 (blue cells). Grade detail in §7 can reconstruct gasoline.
- **2025 product regrouping:** `Raffinerieeinsatzmaterial`, `Spezialbenzine`, `Weitere, nicht aufgeführte Produkte` replace older component lines.
- **Bio detail:** pre-2025 usually one biodiesel aggregate; 2025 PDFs add `davon HVO` / `davon FAME`.
- **Bio labels drift** across XLSX vintages (`Beimischung Bioethanol` vs `Bioethanol`).
- **CID PDFs:** umlauts need fixups; numbers parse fine after decode.

## 4. Broader download grid (end-2019 → latest)

Download every month we can resolve. Gaps here tell us whether Infothek/URL coverage is complete before we invest in a production bootstrap.

In [18]:
# Full history target for v1. Re-run is cheap: cached files are reused.
# End month is "today" so newly published BAFA months are picked up on refresh.
ALL_MONTHS = month_grid("2019-12", pd.Timestamp.today().strftime("%Y-%m"))
all_files = download_many(ALL_MONTHS, PROBE_DIR)

coverage = pd.DataFrame(
    {
        "date": pd.to_datetime([f"{y}-{m:02d}-01" for y, m in ALL_MONTHS]),
        "expected": True,
    }
).merge(
    pd.DataFrame(
        {
            "date": pd.to_datetime([f"{f.year}-{f.month:02d}-01" for f in all_files]),
            "kind": [f.kind for f in all_files],
            "bytes": [f.path.stat().st_size for f in all_files],
        }
    ),
    on="date",
    how="left",
)
coverage["got"] = coverage["kind"].notna()
print(
    f"Downloaded {coverage['got'].sum()} / {len(coverage)} months "
    f"({coverage['got'].mean():.0%})"
)
print("By kind:")
print(coverage["kind"].value_counts(dropna=False))
missing = coverage.loc[~coverage["got"], "date"]
if len(missing):
    print("Missing months:")
    print(", ".join(d.strftime("%Y-%m") for d in missing))

Download gaps (3):
 - No BAFA file for 2026-05 (last error: None)
 - No BAFA file for 2026-06 (last error: None)
 - No BAFA file for 2026-07 (last error: None)
Downloaded 77 / 80 months (96%)
By kind:
kind
pdf     39
xlsx    38
NaN      3
Name: count, dtype: int64
Missing months:
2026-05, 2026-06, 2026-07


In [19]:
fig = px.scatter(
    coverage.assign(y=1),
    x="date",
    y="y",
    color="kind",
    title="BAFA monthly file coverage (probe download)",
    hover_data=["bytes"],
)
fig.update_yaxes(visible=False)
fig.update_layout(height=220, margin=dict(t=50, b=40))
fig.show()

## 5. Parse the full probe set + substitution sketch

Build a tidy panel, then a first-look substitution chart:

- diesel total (blended product from demand)
- biodiesel / FAME+HVO components from bio table
- implied fossil diesel ≈ diesel − biodiesel aggregate (rough; confirm footnote definition in v1)

In [20]:
parts = []
failures = []
for probe in all_files:
    try:
        parts.append(parse_probe_file(probe))
    except Exception as exc:
        failures.append((probe.year, probe.month, str(exc)))

long = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
print(f"Parsed rows: {len(long):,} from {len(parts)} files; failures={len(failures)}")
if failures[:10]:
    print("Sample failures:", failures[:10])

summary = (
    long.groupby(["metric_type", "format_kind"])
    .agg(rows=("value", "size"), products=("product_native", "nunique"), months=("date", "nunique"))
    .reset_index()
)
summary

Parsed rows: 3,620 from 77 files; failures=0


,metric_type,format_kind,rows,products,months
0,BIOBLEND,pdf,156,7,39
1,BIOBLEND,xlsx,114,3,38
2,CLOSTLV,pdf,775,30,39
3,CLOSTLV,xlsx,874,23,38
4,TOTDEMO,pdf,827,40,39
5,TOTDEMO,xlsx,874,23,38


In [21]:
def _norm_bio_label(label: str) -> str:
    s = label.lower()
    if "fame" in s and "hvo" in s and "davon" not in s:
        return "biodiesel_aggregate"
    if "davon hvo" in s or s.strip() == "davon hvo":
        return "hvo"
    if "davon fame" in s:
        return "fame"
    if "bioethanol" in s and "etbe" in s:
        return "bioethanol_etbe"
    if "bioethanol" in s:
        return "bioethanol"
    if "bioheiz" in s:
        return "bio_heating_oil"
    return label


demand = long[long["metric_type"] == "TOTDEMO"].copy()
bio = long[long["metric_type"] == "BIOBLEND"].copy()
bio["bio_key"] = bio["product_native"].map(_norm_bio_label)

diesel = (
    demand[demand["product_native"].str.contains(r"^Dieselkraftstoff$", case=False, na=False)]
    [["date", "value"]]
    .rename(columns={"value": "diesel_total_t"})
)
biodiesel = (
    bio[bio["bio_key"] == "biodiesel_aggregate"][["date", "value"]]
    .rename(columns={"value": "biodiesel_t"})
)
otto = (
    demand[demand["product_native"].str.contains(r"^Ottokraftstoff$", case=False, na=False)]
    [["date", "value"]]
    .rename(columns={"value": "gasoline_total_t"})
)
ethanol = (
    bio[bio["bio_key"] == "bioethanol"][["date", "value"]]
    .rename(columns={"value": "bioethanol_t"})
)

sub = diesel.merge(biodiesel, on="date", how="outer").merge(otto, on="date", how="outer").merge(ethanol, on="date", how="outer")
sub = sub.sort_values("date")
sub["fossil_diesel_implied_t"] = sub["diesel_total_t"] - sub["biodiesel_t"]
sub["biodiesel_share"] = sub["biodiesel_t"] / sub["diesel_total_t"]
sub.tail(12)

,date,diesel_total_t,biodiesel_t,gasoline_total_t,bioethanol_t,fossil_diesel_implied_t,biodiesel_share
65,2025-05-01,2753044.0,190091.0,NaN,98993.0,2562953.0,0.069048
66,2025-06-01,2545982.0,194261.0,NaN,95391.0,2351721.0,0.076301
67,2025-07-01,2992241.0,193849.0,NaN,101681.0,2798392.0,0.064784
68,2025-08-01,2814181.0,179020.0,NaN,86905.0,2635161.0,0.063614
69,2025-09-01,2993721.0,186232.0,NaN,108894.0,2807489.0,0.062208
70,2025-10-01,2977493.0,195677.0,NaN,117616.0,2781816.0,0.065719
71,2025-11-01,2730759.0,189002.0,NaN,110470.0,2541757.0,0.069212
72,2025-12-01,2766126.0,190214.0,NaN,119355.0,2575912.0,0.068765
73,2026-01-01,2337536.0,173021.0,NaN,80921.0,2164515.0,0.074019
74,2026-02-01,2340394.0,203315.0,NaN,81880.0,2137079.0,0.086872


In [22]:
plot_df = sub.melt(
    id_vars="date",
    value_vars=["diesel_total_t", "biodiesel_t", "fossil_diesel_implied_t"],
    var_name="series",
    value_name="tonnes",
)
fig = px.line(
    plot_df,
    x="date",
    y="tonnes",
    color="series",
    title="Germany — diesel total vs biodiesel blend component (probe)",
)
fig.update_layout(legend_title_text="", height=420)
fig.show()

fig2 = px.line(
    sub,
    x="date",
    y="biodiesel_share",
    title="Biodiesel component share of diesel deliveries (probe)",
)
fig2.update_yaxes(tickformat=".1%")
fig2.show()

In [23]:
# Quick diesel box kept as a one-product sanity check; full panels are in §6.
seas = diesel.dropna().copy()
seas["month"] = seas["date"].dt.month
fig = px.box(
    seas,
    x="month",
    y="diesel_total_t",
    points="all",
    title="Diesel deliveries — month box (tonnes, probe sanity check)",
)
fig.show()
print("→ See section 6 for Norway-style seasonality small-multiples (kbd).")

→ See section 6 for Norway-style seasonality small-multiples (kbd).


### Upsert newly published months (no full re-parse)

Run this when BAFA posts new months (e.g. 2026). Downloads only missing/new files, parses them, and replaces overlapping dates in `long`. Then re-run §6 seasonality.

In [24]:
# Incremental refresh — download/parse only months after current `long` max.
# If `long` is missing (fresh kernel), rebuild once from cached probe files (no re-download).
key_cols = ["date", "metric_type", "product_native"]

if "long" not in globals() or long is None or getattr(long, "empty", True):
    print("`long` missing — rebuilding from cached probe files…")
    cached = download_many(month_grid("2019-12", pd.Timestamp.today().strftime("%Y-%m")), PROBE_DIR)
    long = pd.concat([parse_probe_file(f) for f in cached], ignore_index=True)
    print(f"Rebuilt long: {len(long):,} rows | {long['date'].min():%Y-%m}…{long['date'].max():%Y-%m}")
else:
    last = pd.Timestamp(long["date"].max())
    start = (last + pd.offsets.MonthBegin(1)).strftime("%Y-%m")
    end = pd.Timestamp.today().strftime("%Y-%m")
    print(f"Upsert window: {start} → {end} (existing max={last.strftime('%Y-%m')})")
    new_months = month_grid(start, end) if start <= end else []
    new_files = download_many(new_months, PROBE_DIR) if new_months else []
    new_parts = [parse_probe_file(p) for p in new_files]
    if not new_parts:
        print("Nothing new to upsert.")
    else:
        new_df = pd.concat(new_parts, ignore_index=True)
        before = len(long)
        long = (
            pd.concat([long, new_df], ignore_index=True)
            .drop_duplicates(subset=key_cols, keep="last")
            .sort_values(key_cols)
            .reset_index(drop=True)
        )
        print(
            f"Upserted {new_df['date'].nunique()} months / {len(new_df)} rows; "
            f"long {before:,} → {len(long):,} | "
            f"span {long['date'].min():%Y-%m}…{long['date'].max():%Y-%m}"
        )
        print("New months:", ", ".join(sorted(new_df["date"].dt.strftime("%Y-%m").unique())))

Upsert window: 2026-05 → 2026-07 (existing max=2026-04)
Download gaps (3):
 - No BAFA file for 2026-05 (last error: None)
 - No BAFA file for 2026-06 (last error: None)
 - No BAFA file for 2026-07 (last error: None)
Nothing new to upsert.


## 6. Seasonality by year (Norway-style)

Same chart helper as the Norway dashboard (`analytics.seasonality_by_year_chart`):

- one panel per **native** product
- x-axis = calendar month, one line per year
- latest year highlighted in red
- values in **kbd** (tonnes → kt → kbd via IEA densities)

Requires the full probe frame `long` from section 5. Re-run from setup if imports are stale.

In [25]:
from reference._germany_bafa_probe import normalize_product_native

# Re-normalise in case `long` was built before alias fixups landed.
demand_native = long.loc[long["metric_type"] == "TOTDEMO", "product_native"].map(
    normalize_product_native
)
bio_native = long.loc[long["metric_type"] == "BIOBLEND", "product_native"].map(
    normalize_product_native
)

cov = pd.DataFrame(
    {
        "product_native": list(SEASONALITY_DEMAND_PRODUCTS),
        "panel": "demand",
        "months": [
            int((demand_native == p).sum()) for p in SEASONALITY_DEMAND_PRODUCTS
        ],
        "label": [DISPLAY_LABELS.get(p, p) for p in SEASONALITY_DEMAND_PRODUCTS],
    }
)
cov_bio = pd.DataFrame(
    {
        "product_native": list(SEASONALITY_BIO_PRODUCTS),
        "panel": "bio",
        "months": [int((bio_native == p).sum()) for p in SEASONALITY_BIO_PRODUCTS],
        "label": [DISPLAY_LABELS.get(p, p) for p in SEASONALITY_BIO_PRODUCTS],
    }
)
coverage_panels = pd.concat([cov, cov_bio], ignore_index=True)
coverage_panels.sort_values(["panel", "months"], ascending=[True, False])

,product_native,panel,months,label
14,Bioethanol,bio,77,Bioethanol
15,Bioethanol an ETBE a),bio,77,Bioethanol in ETBE
16,"Biodiesel (FAME), HVO, BTL",bio,77,Biodiesel (FAME/HVO/BTL)
17,davon HVO,bio,16,of which HVO
18,davon FAME,bio,16,of which FAME
19,Bioheizöl,bio,6,Bio heating oil
1,Dieselkraftstoff,demand,77,Diesel
2,"Heizöl, leicht",demand,77,Light heating oil
3,"Heizöl, schwer",demand,77,Heavy fuel oil
4,Flüssiggas,demand,77,LPG


In [26]:
def plot_germany_seasonality(metric_type: str = "TOTDEMO") -> None:
    """Norway-style small-multiples; metric_type is TOTDEMO or BIOBLEND."""
    season_df, product_col, products, labels, suffix = seasonality_chart_inputs(
        long, metric_type=metric_type
    )
    if season_df.empty or not products:
        print(f"[skip] No rows for {metric_type} seasonality.")
        return
    fig = seasonality_by_year_chart(
        season_df,
        products,
        value_col="value_kbd",
        date_col="date",
        product_col=product_col,
        product_labels=labels,
        units_label="kbd",
        title=f"Germany BAFA — seasonality ({suffix})",
        default_visible_prior_years=5,
    )
    fig.show()
    print(f"Panels: {len(products)} | rows: {len(season_df):,} | unit: kbd")


plot_germany_seasonality("TOTDEMO")

Panels: 14 | rows: 901 | unit: kbd


In [27]:
# Bio blend components (not additive to §6 diesel/gasoline totals)
plot_germany_seasonality("BIOBLEND")

Panels: 6 | rows: 269 | unit: kbd


## 7. Phase-0 verdict checklist

Use this after running the cells above:

- [ ] Coverage from 2019-12 is high enough (note any missing months)
- [ ] XLSX path yields stable Tab 6c / 8 / 9
- [ ] Modern PDF path yields §6 / §8 / §9
- [ ] CID PDF months are usable (or list them as special-case)
- [ ] Diesel + biodiesel series look continuous enough for substitution charts
- [ ] Otto confidentiality gaps are rare / recoverable via §7
- [ ] Section 6 seasonality panels look sensible (heating oil winter peak, jet summer peak, etc.)

### Likely v1 design (next step, after you confirm)

1. Infothek crawler + content sniff → `data/raw/germany/`
2. Dual parsers: `parse_xlsx_*` and `parse_pdf_*` promoted from the probe module
3. Processor upsert → parquet → optional DuckDB wire-up
4. `product_map.csv` Source=`BAFA` with era-aware native labels
5. Dashboard / notebook for seasonality + substitution panels

**Do not start production wiring until this notebook’s coverage/parse summary looks acceptable.**